# 🫁 RadiAssist: Chest X-ray Triage, full training on Kaggle (GPU T4 x2)

Kaggle port of `colab_train.ipynb`. Differences from the Colab version:

* **No Google Drive.** Results go to `/kaggle/working/cxr-triage` and are saved as this notebook's **Output** when the run finishes. Large temporary files (dataset cache, code) live in `/tmp`.
* **No 42 GB download.** The NIH ChestX-ray14 dataset (`nih-chest-xrays/data`) is attached as an input and read in place.
* **Two GPUs.** Lightning trains with DDP, one process per T4. `train.batch_size` is **per GPU**, so 64 means an effective batch of 128. A patch cell makes validation AUROC use the predictions from both GPUs.
* **Not run here:** Hugging Face publishing (needs a secret). Download the `bundle/` folder from the Output and publish it from Colab or locally.

## Step 0: check the GPUs

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() == 2, "Expected GPU T4 x2: set Accelerator to 'GPU T4 x2'"

## Step 1: code and dependencies

In [ ]:
import os, sys
WORK = "/kaggle/working/cxr-triage"   # saved as notebook output
SCRATCH = "/tmp/cxr"                  # big temporary files, not saved
os.makedirs(WORK, exist_ok=True)
os.makedirs(SCRATCH, exist_ok=True)

REPO_URL = "https://github.com/dhiman-13/chest-xray-triage.git"
PROJECT = f"{SCRATCH}/chest-xray-triage"
if not os.path.exists(PROJECT):
    !git clone -q {REPO_URL} {PROJECT}
%cd {PROJECT}
!git log -1 --format="%h %s"

In [ ]:
# torch / torchvision are already installed on Kaggle and are not re-downloaded
!pip install -q -r requirements.txt
!pip install -q -e .
# Make the project importable both here and in every `!python -m cxr...` subprocess
os.environ["PYTHONPATH"] = f"{PROJECT}/src:{PROJECT}"
for p in (f"{PROJECT}/src", PROJECT):
    if p not in sys.path:
        sys.path.insert(0, p)
!python -c "import cxr, timm, lightning, hydra, mlflow, cv2, onnx; print('All packages OK')"

## Step 1b: make training safe on 2 GPUs (DDP)
With DDP each GPU validates half of the validation set. Patch 1 gathers both halves before computing AUROC, so checkpointing and early stopping see the real macro-AUROC on every process. Patch 2 lets only the main process write `last_run.json`.
The repo includes both fixes from the `kaggle-t4x2-training` branch on; for a checkout without them this cell patches the cloned copy, otherwise it only prints "already patched".

In [ ]:
from pathlib import Path

def patch(path, old, new):
    p = Path(PROJECT) / path
    s = p.read_text()
    if new in s:
        print("already patched:", path)
        return
    assert old in s, f"Patch anchor not found in {path}: the repo changed, update this cell"
    p.write_text(s.replace(old, new, 1))
    print("patched:", path)

patch("src/cxr/models/classifier.py",
"""        p = torch.sigmoid(torch.cat(self._val_logits)).numpy()
        y = torch.cat(self._val_targets).numpy()""",
"""        logits, targets = torch.cat(self._val_logits), torch.cat(self._val_targets)
        if self.trainer.world_size > 1:  # DDP: gather every GPU's share of the validation set
            logits = self.all_gather(logits.to(self.device)).flatten(0, 1).cpu()
            targets = self.all_gather(targets.to(self.device)).flatten(0, 1).cpu()
        p = torch.sigmoid(logits).numpy()
        y = targets.numpy()""")

patch("src/cxr/train.py",
"""    (out_dir / "last_run.json").write_text(json.dumps(run_info, indent=2))
    logger.experiment.log_metric(logger.run_id, "best_val_auroc_macro", best)
    print(json.dumps(run_info, indent=2))""",
"""    if trainer.is_global_zero:  # DDP: only the main process records the run
        (out_dir / "last_run.json").write_text(json.dumps(run_info, indent=2))
        logger.experiment.log_metric(logger.run_id, "best_val_auroc_macro", best)
        print(json.dumps(run_info, indent=2))""")

## Step 2: dataset (NIH ChestX-ray14)
112,120 frontal X-rays of 30,805 patients with 14 labels, read straight from the attached Kaggle dataset, resized to 256x256 and packed into one `images.npy` (~7 GB in `/tmp`).
It uses the **official patient-wise test split**: no patient appears in both train and test.

In [ ]:
import glob
SUBSET = 1.0
tag = "full" if SUBSET >= 1 else f"subset{int(SUBSET*100)}"
DATA = f"{SCRATCH}/data/processed_{tag}"

found = glob.glob("/kaggle/input/**/Data_Entry_2017.csv", recursive=True)
assert found, "NIH dataset not attached: add the input 'nih-chest-xrays/data'"
NIH = os.path.dirname(found[0])
print("NIH dataset:", NIH)

if not os.path.exists(f"{DATA}/images.npy"):
    !python -m cxr.data.prepare_nih --source {NIH} --out {DATA} --subset {SUBSET} --workers {os.cpu_count()}
assert os.path.exists(f"{DATA}/images.npy"), "Preparation failed, see the error above"
!cp {DATA}/summary.json {WORK}/dataset_summary.json
!head -c 1200 {DATA}/summary.json

## Step 3: look at the data

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from cxr.labels import LABELS
meta = pd.read_csv(f"{DATA}/metadata.csv")
images = np.load(f"{DATA}/images.npy", mmap_mode="r")
print(meta["split"].value_counts().to_string(), "\n")
print("patients per split:", meta.groupby("split")["patient_id"].nunique().to_dict())

fig, ax = plt.subplots(1, 2, figsize=(15, 4))
prev = meta[meta.split == "train"][LABELS].mean().sort_values()
prev.plot.barh(ax=ax[0], color="#0d9488"); ax[0].set_title("Label prevalence (train): heavy class imbalance")
meta[LABELS].sum(1).value_counts().sort_index().plot.bar(ax=ax[1], color="#6366f1")
ax[1].set_title("Findings per image (0 = 'No Finding')"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, (_, r) in zip(axes.ravel(), meta.sample(10, random_state=1).iterrows()):
    ax.imshow(images[r.idx], cmap="gray"); ax.axis("off")
    ax.set_title(r.findings.replace("|", ", ")[:30] + f"\n{r.sex}, {r.age:.0f}y, {r.view}", fontsize=8)
plt.tight_layout(); plt.show()

## Step 4: experiment tracking (local MLflow, saved in the Output)

In [ ]:
os.makedirs(f"{WORK}/mlflow", exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"sqlite:///{WORK}/mlflow/mlflow.db"
ARTIFACTS = f"{WORK}/mlflow/artifacts"

OUT = f"{WORK}/outputs"        # checkpoints + last_run.json
REPORTS = f"{WORK}/reports"    # evaluation results
BUNDLE = f"{WORK}/bundle"      # deployable model
print(os.environ["MLFLOW_TRACKING_URI"])

## Step 5: sanity run (~5 min)
3% of the data for one epoch, on both GPUs. If this crashes, the long run would too.

In [ ]:
!python -m cxr.train data.dir={DATA} paths.output_dir={SCRATCH}/sanity mlflow.experiment=cxr-sanity \
    mlflow.artifact_location={ARTIFACTS} train.epochs=1 train.limit_train_batches=0.03 train.limit_val_batches=0.1 \
    train.num_workers=2
assert os.path.exists(f"{SCRATCH}/sanity/last_run.json"), "Sanity run failed, see the error above"

## Step 6: full training (DDP on 2x T4)
DenseNet-121 pretrained on ImageNet (the CheXNet recipe), AdamW + warm-up + cosine LR, 16-bit mixed precision, class-weighted BCE, early stopping on validation macro-AUROC.
`batch_size=64` is per GPU (effective 128). `num_workers=2` per GPU uses Kaggle's 4 CPUs.

In [ ]:
!python -m cxr.train data.dir={DATA} paths.output_dir={OUT} mlflow.artifact_location={ARTIFACTS} \
    model=densenet121 train.epochs=12 train.batch_size=64 train.num_workers=2
assert os.path.exists(f"{OUT}/last_run.json"), "Training failed, see the error above"
!cat {OUT}/last_run.json

## Step 7: evaluation
* Per-class **AUROC with 95% bootstrap confidence intervals**, AUPRC, sensitivity at 90% specificity
* **Temperature scaling** (calibration) and **per-class thresholds for 90% sensitivity**, both fit on *validation* data only
* **Subgroup analysis** (sex, age, view), **Grad-CAM** heatmaps, comparison with published baselines

In [ ]:
!python -m cxr.evaluate --run-info {OUT}/last_run.json --out {REPORTS}
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{REPORTS}/RESULTS.md").read().split("![")[0]))
for f in ["auroc_vs_baselines", "roc_curves", "reliability", "gradcam"]:
    display(Image(f"{REPORTS}/figures/{f}.png"))

## Step 8: export, register, promote
* **Export:** ONNX graph with calibrated probabilities and class-activation heatmaps built in; parity-checked against PyTorch
* **Register:** a new version in the MLflow Model Registry (alias `challenger`)
* **Promote:** it becomes `champion` only if its test AUROC beats the current champion (quality gate)

In [ ]:
!python -m cxr.export --run-info {OUT}/last_run.json --reports {REPORTS} --out {BUNDLE}
!python -m cxr.registry register --run-info {OUT}/last_run.json --bundle {BUNDLE}
!python -m cxr.registry promote --run-info {OUT}/last_run.json
!ls -lh {BUNDLE}

## Step 9: test the real API on a test X-ray

In [ ]:
!pip install -q fastapi python-multipart prometheus-client httpx "uvicorn[standard]"
os.environ["MODEL_DIR"] = BUNDLE
os.environ["DB_PATH"] = f"{SCRATCH}/predictions.db"
import cv2
from fastapi.testclient import TestClient
from serving.app.main import app

row = meta[(meta.split == "test") & (meta.Effusion == 1)].iloc[0]
ok, png = cv2.imencode(".png", np.asarray(images[row.idx]))
with TestClient(app) as client:
    r = client.post("/api/predict", files={"file": ("xray.png", png.tobytes(), "image/png")}).json()
print("Ground truth:", row.findings)
print("Triage:", r["triage"].upper(), "| flagged:", r["flagged"], "| latency:", r["latency_ms"], "ms")
for f in r["findings"][:5]:
    print(f"  {f['label']:20s} p={f['probability']:.3f}  threshold={f['threshold']:.3f}")

## Done 🎉
Everything is in this notebook's **Output** tab under `cxr-triage/`:
* `reports/RESULTS.md` and `reports/figures/`: results for your CV/README
* `bundle/`: the deployable ONNX model (publish it to Hugging Face with `cxr.registry publish` and `deploy/deploy_space.py`)
* `outputs/checkpoints/`: Lightning checkpoints; `mlflow/`: the MLflow database (`mlflow ui --backend-store-uri sqlite:///mlflow/mlflow.db`)

In [ ]:
!du -sh {WORK}/*